# DfT data audit and preparation
BUSI 1783 — AI Assisted Traffic Signal Control

This notebook inspects the supplied Birmingham and Manchester CSV files. It documents coverage, checks data quality and prepares a **2020–2025 screening subset**. The final selection of road networks and observations will depend on data availability and quality. No AI controller has been trained and no SUMO simulation has been run.


## Sources
- Birmingham: https://roadtraffic.dft.gov.uk/local-authorities/E08000025
- Manchester: https://roadtraffic.dft.gov.uk/local-authorities/E08000003
- Metadata and guidance: https://roadtraffic.dft.gov.uk/downloads
- Contains public sector information licensed under the Open Government Licence v3.0: https://www.nationalarchives.gov.uk/doc/open-government-licence/version/3/

Files supplied for review on 5 October 2026. 

DfT counts and planned OpenStreetMap geometry are secondary inputs. Future SUMO trajectories will be computational outputs derived from those inputs and documented assumptions. Counts alone do not measure queues, delay or signal timings.

In [1]:
from pathlib import Path
import hashlib
import platform
import pandas as pd
from IPython.display import display

cwd = Path.cwd()
ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'data').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Place the notebook in the repository with a data folder.')
DATA = ROOT / 'data'
SCREEN_START, SCREEN_END = 2020, 2025
print('Python:', platform.python_version(), '| pandas:', pd.__version__)
print('Screening window:', SCREEN_START, 'to', SCREEN_END)

raw, points, provenance = {}, {}, []
for city, authority in [('Birmingham', 141), ('Manchester', 85)]:
    for kind, store in [('rawcount', raw), ('countpoints', points)]:
        matches = sorted(DATA.glob(f'dft_{kind}_local_authority_id_{authority}*.csv'))
        if len(matches) != 1:
            raise ValueError(f'{city} {kind}: expected one CSV, found {len(matches)}')
        path = matches[0]
        frame = pd.read_csv(path, low_memory=False)
        assert frame['local_authority_id'].eq(authority).all(), 'Unexpected authority'
        store[city] = frame
        provenance.append({'city': city, 'kind': kind, 'filename': path.name,
                           'rows': len(frame), 'sha256': hashlib.sha256(path.read_bytes()).hexdigest()})
display(pd.DataFrame(provenance)[['city', 'kind', 'filename', 'rows']])

Python: 3.12.14 | pandas: 2.2.3
Screening window: 2020 to 2025


,city,kind,filename,rows
0,Birmingham,rawcount,dft_rawcount_local_authority_id_141_BIR.csv,72948
1,Birmingham,countpoints,dft_countpoints_local_authority_id_141 BIR.csv,590
2,Manchester,rawcount,dft_rawcount_local_authority_id_85_MANCHESTER.csv,39072
3,Manchester,countpoints,dft_countpoints_local_authority_id_85_ MANCHES...,342


## 1. Year coverage
Each raw-count row records a count point, direction, date and hour. The location tables have no year column. Presence of a year in a city does not establish annual coverage at every road. Row totals are survey coverage measures, not city-wide annual traffic volumes.

This section audits the full source history (2000–2025). Section 3 applies the selected 2020–2025 period. Earlier source records are retained for provenance but excluded from that subset.

In [2]:
coverage, annual = [], []
for city, df in raw.items():
    years = sorted(df['year'].dropna().astype(int).unique())
    coverage.append({'city': city, 'raw_rows': len(df), 'first_year': min(years),
                     'last_year': max(years), 'years_present': len(years),
                     'missing_years': sorted(set(range(min(years), max(years)+1))-set(years)),
                     'location_rows': len(points[city]),
                     'raw_count_points': df['count_point_id'].nunique()})
    a = df.groupby('year').agg(rows=('count_point_id', 'size'),
                             count_points=('count_point_id', 'nunique'),
                             survey_dates=('count_date', 'nunique')).reset_index()
    a.insert(0, 'city', city)
    annual.append(a)
display(pd.DataFrame(coverage))
annual_coverage = pd.concat(annual, ignore_index=True)
display(annual_coverage.pivot(index='year', columns='city', values='rows'))

,city,raw_rows,first_year,last_year,years_present,missing_years,location_rows,raw_count_points
0,Birmingham,72948,2000,2025,26,[],590,562
1,Manchester,39072,2000,2025,26,[],342,305


city,Birmingham,Manchester
year,,
2000,1872,1368
2001,2220,1632
2002,2076,1464
2003,2880,1584
2004,2664,1296
2005,2448,1524
2006,2628,1452
2007,2676,1308
2008,5844,2460


## 2. Quality checks
Missing values are retained. Junction names and link lengths may be absent, particularly on minor roads; they are not replaced with zero. Duplicate keys, unmatched locations and arithmetic inconsistencies require review before demand calibration.

In [3]:
quality, missing_tables = [], []
key = ['count_point_id', 'direction_of_travel', 'count_date', 'hour']
vehicle_cols = ['two_wheeled_motor_vehicles', 'cars_and_taxis', 'buses_and_coaches', 'lgvs', 'all_hgvs']
for city, df in raw.items():
    dates = pd.to_datetime(df['count_date'], errors='coerce')
    pts = points[city]
    quality.append({'city': city,
        'exact_duplicate_rows': int(df.duplicated().sum()),
        'duplicate_observation_keys': int(df.duplicated(key).sum()),
        'invalid_dates': int(dates.isna().sum()),
        'year_date_mismatches': int((dates.dt.year != df['year']).sum()),
        'missing_coordinates': int(df[['latitude','longitude']].isna().any(axis=1).sum()),
        'negative_vehicle_counts': int((df[vehicle_cols+['all_motor_vehicles']] < 0).any(axis=1).sum()),
        'vehicle_total_mismatches': int((df[vehicle_cols].sum(axis=1, min_count=5) != df['all_motor_vehicles']).sum()),
        'duplicate_location_ids': int(pts['count_point_id'].duplicated().sum()),
        'raw_rows_without_location': int((~df['count_point_id'].isin(pts['count_point_id'])).sum())})
    for kind, frame in [('raw counts', df), ('count points', pts)]:
        m = frame.isna().sum()
        for column, count in m[m > 0].items():
            missing_tables.append({'city': city, 'table': kind, 'column': column,
                                   'missing_rows': int(count), 'percent': round(100*count/len(frame), 2)})
display(pd.DataFrame(quality).set_index('city').T)
display(pd.DataFrame(missing_tables))
print('Observed hours by city:', {city: sorted(df['hour'].unique().tolist()) for city, df in raw.items()})

city,Birmingham,Manchester
exact_duplicate_rows,0,0
duplicate_observation_keys,0,0
invalid_dates,0,0
year_date_mismatches,0,0
missing_coordinates,0,0
negative_vehicle_counts,0,0
vehicle_total_mismatches,0,0
duplicate_location_ids,0,0
raw_rows_without_location,48,0


,city,table,column,missing_rows,percent
0,Birmingham,raw counts,start_junction_road_name,49680,68.10
1,Birmingham,raw counts,end_junction_road_name,49680,68.10
2,Birmingham,raw counts,link_length_km,49680,68.10
3,Birmingham,raw counts,link_length_miles,49680,68.10
4,Birmingham,count points,start_junction_road_name,394,66.78
5,Birmingham,count points,end_junction_road_name,394,66.78
6,Birmingham,count points,link_length_km,394,66.78
7,Birmingham,count points,link_length_miles,394,66.78
8,Manchester,raw counts,start_junction_road_name,18504,47.36
9,Manchester,raw counts,end_junction_road_name,18504,47.36


Observed hours by city: {'Birmingham': [7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18], 'Manchester': [7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]}


## 3. Prepare the 2020–2025 screening subset
Retain all original columns and rows within the chosen years. Do not pool counts across different roads as a demand profile. The tables below identify available locations for later map inspection. They do not establish that a road has suitable signal-controlled junctions.

The match uses a unique count-point ID and preserves the historical coordinates in the raw observations. Location-table coordinates receive a separate prefix because road metadata may change over time.

In [4]:
screened, candidates = {}, []
for city, df in raw.items():
    subset = df.loc[df['year'].between(SCREEN_START, SCREEN_END)].copy()
    pts = points[city][['count_point_id', 'latitude', 'longitude']].rename(
        columns={'latitude': 'location_latitude', 'longitude': 'location_longitude'})
    subset = subset.merge(pts, on='count_point_id', how='left', validate='many_to_one', indicator='location_match')
    screened[city] = subset
    candidate = subset.groupby('count_point_id').agg(
        road_name=('road_name', lambda x: ', '.join(sorted(x.dropna().astype(str).unique()))),
        first_year=('year', 'min'), last_year=('year', 'max'), years_observed=('year', 'nunique'),
        survey_dates=('count_date', 'nunique'), hourly_rows=('hour', 'size'),
        latitude=('latitude', 'first'), longitude=('longitude', 'first')).reset_index()
    candidate.insert(0, 'city', city)
    candidates.append(candidate)
    print(city, ':', len(subset), 'rows;', subset['count_point_id'].nunique(), 'count points')
candidate_points = pd.concat(candidates, ignore_index=True)
display(candidate_points.sort_values(['city','years_observed','hourly_rows'], ascending=[True,False,False]).groupby('city').head(10))

Birmingham : 11040 rows; 178 count points
Manchester : 7380 rows; 133 count points


,city,count_point_id,road_name,first_year,last_year,years_observed,survey_dates,hourly_rows,latitude,longitude
38,Birmingham,36070,"A38(M), A38M",2020,2025,6,6,144,52.507305,-1.876193
115,Birmingham,802957,U,2020,2025,6,6,144,52.470231,-1.951931
123,Birmingham,802968,U,2020,2025,6,6,144,52.405835,-2.012369
125,Birmingham,802970,U,2020,2025,6,6,144,52.463268,-1.962565
129,Birmingham,802977,U,2020,2025,6,6,144,52.549908,-1.904594
132,Birmingham,802982,U,2020,2025,6,6,144,52.430918,-1.933692
140,Birmingham,805330,U,2020,2025,6,6,144,52.452476,-1.985265
153,Birmingham,807738,U,2020,2025,6,6,144,52.409848,-1.871509
154,Birmingham,807907,U,2020,2025,6,6,144,52.540127,-1.894043
155,Birmingham,808038,C,2020,2025,6,6,144,52.458279,-1.925793


## 4. Optional exports
Set `EXPORT = True` to save audit summaries and screened observations under `outputs/data_audit/`. Original source CSVs remain unchanged. These exports are processed observational data, not SUMO outputs.

In [5]:
EXPORT = True
if EXPORT:
    out = ROOT / 'outputs' / 'data_audit'
    out.mkdir(parents=True, exist_ok=True)
    annual_coverage.to_csv(out / 'annual_coverage.csv', index=False)
    pd.DataFrame(quality).to_csv(out / 'quality_checks.csv', index=False)
    pd.DataFrame(provenance).to_csv(out / 'source_manifest.csv', index=False)
    candidate_points.to_csv(out / 'candidate_count_points.csv', index=False)
    for city, frame in screened.items():
        frame.to_csv(out / f'{city.lower()}_screened_counts.csv', index=False)
    print('Exports saved in outputs/data_audit')
else:
    print('Exports disabled; source files have not been changed.')

Exports disabled; source files have not been changed.
